# Assignment 02: Backpropagation by Hand (100 points)

**Unit 07: Deep Learning | AI 410**

In this assignment, you will trace the backward pass through small networks with concrete numbers, verifying your hand computations against PyTorch autograd. This is a critical USAAIO skill — Round 1 frequently tests manual gradient computation.

**Notation**:
- $\delta^{[l]} = \frac{\partial L}{\partial z^{[l]}}$ (error signal at layer $l$)
- $\odot$ = element-wise (Hadamard) product
- All gradients are computed for **single samples** (no batch averaging)

In [ ]:
"""DO NOT MAKE ANY CHANGE IN THIS CELL."""
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)

**WARNING**: Do not import any additional libraries. You must compute gradients by hand (as tensors), then verify against autograd.

---

## Part 1 (20 points, coding)

**Single neuron with sigmoid**: $\hat{y} = \sigma(wx + b)$, $L = (\hat{y} - y)^2$

Given: $w = 0.5$, $b = -0.1$, $x = 2.0$, $y = 1.0$

Compute each value as a Python float:
1. `z` = pre-activation
2. `y_hat` = $\sigma(z)$
3. `loss` = $(\hat{y} - y)^2$
4. `dL_dy_hat` = $\frac{\partial L}{\partial \hat{y}}$
5. `dL_dz` = $\frac{\partial L}{\partial z}$ (use $\sigma'(z) = \sigma(z)(1-\sigma(z))$)
6. `dL_dw` = $\frac{\partial L}{\partial w}$
7. `dL_db` = $\frac{\partial L}{\partial b}$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Given values
w_val, b_val, x_val, y_val = 0.5, -0.1, 2.0, 1.0

# Forward pass
z = ...           # w*x + b
y_hat = ...       # sigmoid(z)
loss = ...        # (y_hat - y)^2

# Backward pass
dL_dy_hat = ...   # 2*(y_hat - y)
dL_dz = ...       # dL_dy_hat * sigmoid'(z)
dL_dw = ...       # dL_dz * x
dL_db = ...       # dL_dz * 1

In [ ]:
""" END OF THIS PART """
# Verify against autograd
w_t = torch.tensor(w_val, requires_grad=True)
b_t = torch.tensor(b_val, requires_grad=True)
x_t = torch.tensor(x_val)
y_t = torch.tensor(y_val)
z_t = w_t * x_t + b_t
yh_t = torch.sigmoid(z_t)
loss_t = (yh_t - y_t) ** 2
loss_t.backward()

assert abs(z - z_t.item()) < 1e-6, f"z: got {z}, expected {z_t.item()}"
assert abs(y_hat - yh_t.item()) < 1e-4, f"y_hat: got {y_hat}, expected {yh_t.item()}"
assert abs(loss - loss_t.item()) < 1e-4, f"loss: got {loss}, expected {loss_t.item()}"
assert abs(dL_dw - w_t.grad.item()) < 1e-4, f"dL/dw: got {dL_dw}, expected {w_t.grad.item()}"
assert abs(dL_db - b_t.grad.item()) < 1e-4, f"dL/db: got {dL_db}, expected {b_t.grad.item()}"
print("Part 1 passed!")

---

## Part 2 (25 points, coding)

**Two-layer network**: $2 \to 2 \to 1$, ReLU hidden, no output activation, MSE loss.

Given:
```
W1 = [[0.1, 0.3],    b1 = [0.0, 0.0]
      [0.2, 0.4]]

W2 = [[0.5, 0.6]]    b2 = [0.0]

x = [1.0, 2.0],  y = 1.0
```

Compute ALL intermediate values and gradients as torch tensors:
- Forward: `z1`, `a1`, `z2`, `y_hat`
- Loss: `loss` = $(\hat{y} - y)^2$
- Backward: `dL_dW2`, `dL_db2`, `dL_dW1`, `dL_db1`

In [ ]:
### WRITE YOUR SOLUTION HERE ###

W1 = torch.tensor([[0.1, 0.3], [0.2, 0.4]])
b1 = torch.tensor([0.0, 0.0])
W2 = torch.tensor([[0.5, 0.6]])
b2 = torch.tensor([0.0])
x = torch.tensor([1.0, 2.0])
y = torch.tensor([1.0])

# Forward pass (compute these)
z1 = ...       # (2,) — W1 @ x + b1
a1 = ...       # (2,) — ReLU(z1)
z2 = ...       # (1,) — W2 @ a1 + b2
y_hat = ...    # (1,) — output (no activation)
loss = ...     # scalar — MSE

# Backward pass (compute these)
dL_dy_hat = ... # (1,)
delta2 = ...    # (1,) — dL/dz2
dL_dW2 = ...    # (1, 2)
dL_db2 = ...    # (1,)
dL_da1 = ...    # (2,) — gradient flowing to hidden layer
delta1 = ...    # (2,) — dL/dz1 (after ReLU gate)
dL_dW1 = ...    # (2, 2)
dL_db1 = ...    # (2,)

In [ ]:
""" END OF THIS PART """
# Verify against autograd
W1_ag = torch.tensor([[0.1, 0.3], [0.2, 0.4]], requires_grad=True)
b1_ag = torch.tensor([0.0, 0.0], requires_grad=True)
W2_ag = torch.tensor([[0.5, 0.6]], requires_grad=True)
b2_ag = torch.tensor([0.0], requires_grad=True)
x_ag = torch.tensor([1.0, 2.0])
y_ag = torch.tensor([1.0])

z1_ag = W1_ag @ x_ag + b1_ag
a1_ag = torch.relu(z1_ag)
z2_ag = W2_ag @ a1_ag + b2_ag
loss_ag = (z2_ag - y_ag).pow(2)
loss_ag.backward()

assert torch.allclose(dL_dW1, W1_ag.grad, atol=1e-5), f"dL/dW1 mismatch"
assert torch.allclose(dL_db1, b1_ag.grad, atol=1e-5), f"dL/db1 mismatch"
assert torch.allclose(dL_dW2, W2_ag.grad, atol=1e-5), f"dL/dW2 mismatch"
assert torch.allclose(dL_db2, b2_ag.grad, atol=1e-5), f"dL/db2 mismatch"
print("Part 2 passed!")

---

## Part 3 (20 points, coding)

**Softmax + Cross-Entropy gradient**: Verify the elegant formula.

Given logits $z = [2.0, 1.0, 0.1]$ and true class $y = 0$:

1. Compute softmax probabilities `probs` from `z` (with numerical stability)
2. Compute cross-entropy loss `loss` = $-\log(p_y)$
3. Compute gradient `dL_dz` using the formula $\frac{\partial L}{\partial z_i} = p_i - \mathbb{1}[i = y]$
4. Verify against autograd

In [ ]:
### WRITE YOUR SOLUTION HERE ###

z = torch.tensor([2.0, 1.0, 0.1])
y_class = 0  # true class

# Step 1: Compute softmax (use log-sum-exp trick for stability)
probs = ...     # (3,) — softmax probabilities

# Step 2: Cross-entropy loss
loss = ...      # scalar — -log(probs[y_class])

# Step 3: Gradient using the elegant formula
dL_dz = ...     # (3,) — probs - one_hot(y_class)

In [ ]:
""" END OF THIS PART """
# Verify probabilities sum to 1
assert abs(probs.sum().item() - 1.0) < 1e-6, "Probabilities must sum to 1"
assert (probs > 0).all(), "All probabilities must be positive"

# Verify against autograd
z_ag = torch.tensor([2.0, 1.0, 0.1], requires_grad=True)
loss_ag = torch.nn.functional.cross_entropy(z_ag.unsqueeze(0), torch.tensor([0]))
loss_ag.backward()

assert abs(loss.item() - loss_ag.item()) < 1e-5, f"Loss mismatch: {loss.item()} vs {loss_ag.item()}"
assert torch.allclose(dL_dz, z_ag.grad, atol=1e-5), f"Gradient mismatch: {dL_dz} vs {z_ag.grad}"
print("Part 3 passed!")

---

## Part 4 (20 points, coding)

**Three-layer network with dead neurons**: Trace the backward pass through a $3 \to 4 \to 2 \to 1$ network with ReLU and identify which gradients are zero due to dead neurons.

Given the weights and input below, compute:
1. Full forward pass (identify which ReLU neurons are active)
2. Full backward pass (MSE loss, target $y = 2.0$)
3. Store `dead_neurons_layer1` (list of indices of dead neurons in layer 1)
4. Store `zero_grad_weights` (list of strings like `'W1[2,:]'` for rows of W1 with zero gradient)

In [ ]:
"""DO NOT MAKE ANY CHANGE IN THIS CELL."""
W1_p4 = torch.tensor([[0.5, -0.3, 0.2],
                       [0.1, 0.4, -0.6],
                       [-0.2, 0.1, 0.3],
                       [0.7, -0.5, 0.1]], dtype=torch.float32)
b1_p4 = torch.tensor([0.1, -0.8, 0.0, 0.2], dtype=torch.float32)
W2_p4 = torch.tensor([[0.3, -0.2, 0.5, 0.1],
                       [-0.4, 0.6, 0.2, -0.3]], dtype=torch.float32)
b2_p4 = torch.tensor([0.0, 0.1], dtype=torch.float32)
W3_p4 = torch.tensor([[0.7, -0.5]], dtype=torch.float32)
b3_p4 = torch.tensor([0.0], dtype=torch.float32)
x_p4 = torch.tensor([1.0, -1.0, 0.5], dtype=torch.float32)
y_p4 = torch.tensor([2.0], dtype=torch.float32)

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Forward pass
z1_p4 = ...   # (4,)
a1_p4 = ...   # (4,) — ReLU
z2_p4 = ...   # (2,)
a2_p4 = ...   # (2,) — ReLU
z3_p4 = ...   # (1,)
y_hat_p4 = ...  # (1,)
loss_p4 = ...   # scalar — (y_hat - y)^2

# Backward pass
dL_dW3 = ...  # (1, 2)
dL_db3 = ...  # (1,)
dL_dW2 = ...  # (2, 4)
dL_db2 = ...  # (2,)
dL_dW1 = ...  # (4, 3)
dL_db1 = ...  # (4,)

# Analysis
dead_neurons_layer1 = ...   # list of int indices, e.g., [1, 3]
dead_neurons_layer2 = ...   # list of int indices

In [ ]:
""" END OF THIS PART """
# Verify with autograd
W1_ag = W1_p4.clone().requires_grad_(True)
b1_ag = b1_p4.clone().requires_grad_(True)
W2_ag = W2_p4.clone().requires_grad_(True)
b2_ag = b2_p4.clone().requires_grad_(True)
W3_ag = W3_p4.clone().requires_grad_(True)
b3_ag = b3_p4.clone().requires_grad_(True)

z1_ag = W1_ag @ x_p4 + b1_ag
a1_ag = torch.relu(z1_ag)
z2_ag = W2_ag @ a1_ag + b2_ag
a2_ag = torch.relu(z2_ag)
z3_ag = W3_ag @ a2_ag + b3_ag
loss_ag = (z3_ag - y_p4).pow(2)
loss_ag.backward()

assert torch.allclose(dL_dW1, W1_ag.grad, atol=1e-4), "dW1 mismatch"
assert torch.allclose(dL_dW2, W2_ag.grad, atol=1e-4), "dW2 mismatch"
assert torch.allclose(dL_dW3, W3_ag.grad, atol=1e-4), "dW3 mismatch"
# Verify dead neuron identification
actual_dead_l1 = [i for i in range(4) if a1_ag[i].item() == 0]
assert sorted(dead_neurons_layer1) == sorted(actual_dead_l1), \
    f"Dead neurons layer 1: expected {actual_dead_l1}, got {dead_neurons_layer1}"
print("Part 4 passed!")

---

## Part 5 (15 points, non-coding)

Answer the following questions about backpropagation in the markdown cell below.

1. In a 10-layer network with sigmoid activations, the gradient at layer 1 is approximately $0.25^9 \approx 3.8 \times 10^{-6}$. Explain why, and describe TWO techniques that solve this.

2. Why does the softmax + cross-entropy gradient simplify to $p - y$? What would the gradient look like if we computed softmax and cross-entropy separately (without combining)?

3. In the network from Part 4, if a neuron in layer 1 is dead for ALL training samples (not just this one input), can it ever recover through gradient descent? Why or why not?

### WRITE YOUR SOLUTION HERE ###

1. *Your answer here*

2. *Your answer here*

3. *Your answer here*

""" END OF THIS PART """